In [1]:
!pip --version

pip 26.2.1 from D:\yugyeong\hanwha_0902\days\ex_0915\.venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model = "gpt-4o-mini",
    max_completion_tokens = 2048,
    temperature = 0.2
)

In [6]:
examples = [
    {
        "question" : "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer" : """이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 스티브 잡스는 몇 살에 사망했나요?
        중간 답변: 스티브 잡스는 56세에 사망했습니다.
        추가 질문: 아인슈타인은 몇 살에 사망했나요?
        중간 답변: 아인슈타인은 76세에 사망했습니다.
        최종 답변은: 아인슈타인
        """
    },
    {
        "question": "네이버의 창립자는 언제 태어났나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 네이버의 창립자는 누구인가요?
        중간 답변: 네이버는 이해진에 의해 창립되었습니다.
        추가 질문: 이해진은 언제 태어났나요?
        중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
        최종 답변은: 1967년 6월 22일
        """,
    },
    {
        "question": "율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 율곡 이이의 어머니는 누구인가요?
        중간 답변: 율곡 이이의 어머니는 신사임당입니다.
        추가 질문: 신사임당은 언제 태어났나요?
        중간 답변: 신사임당은 1504년에 태어났습니다.
        추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
        중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
        최종 답변은: 연산군
        """,
    },
    {
        "question": "올드보이와 기생충의 감독이 같은 나라 출신인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 올드보이의 감독은 누구인가요?
        중간 답변: 올드보이의 감독은 박찬욱입니다.
        추가 질문: 박찬욱은 어느 나라 출신인가요?
        중간 답변: 박찬욱은 대한민국 출신입니다.
        추가 질문: 기생충의 감독은 누구인가요?
        중간 답변: 기생충의 감독은 봉준호입니다.
        추가 질문: 봉준호는 어느 나라 출신인가요?
        중간 답변: 봉준호는 대한민국 출신입니다.
        최종 답변은: 예
        """,
    },
    
]

In [21]:
# SemanticSimilarityExampleSelector 현재 질문과 의미가 가장 비슷한 예시 선택
from langchain_core.example_selectors import MaxMarginalRelevanceExampleSelector, SemanticSimilarityExampleSelector
# OpenAIEmbeddings 문장을 벡터로 변환
from langchain_openai import OpenAIEmbeddings
# Chroma 벡터DB
from langchain_chroma import Chroma
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# Vector DB 생성 (저장소 이름, 임베딩 클래스(어떤 방식으로 벡터화할지 지정))
# chroma = Chroma("example_selector", OpenAIEmbeddings())

example_selector = SemanticSimilarityExampleSelector.from_examples(
    # 검색할 예시 모음
    examples,
    
    # 예시를 벡터로 변환
    OpenAIEmbeddings(),
    
    # 벡터를 저장하고 검색
    Chroma,
    
    # 가장 비슷한 예시 1개만 가져오기
    k=1,
)

question = "Google 창립 연도 빌게이츠 나이는 몇 살인가요?"

selected_examples = example_selector.select_examples({"question": question})
print(f"입력에 가장 유사한 예시:\n{question}\n")

for example in selected_examples:
    print(f'question:\n{example["question"]}')
    print(f'answer:\n{example["answer"]}')

입력에 가장 유사한 예시:
Google 창립 연도 빌게이츠 나이는 몇 살인가요?

question:
네이버의 창립자는 언제 태어났나요?
answer:
이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 네이버의 창립자는 누구인가요?
        중간 답변: 네이버는 이해진에 의해 창립되었습니다.
        추가 질문: 이해진은 언제 태어났나요?
        중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
        최종 답변은: 1967년 6월 22일
        


In [22]:
example_prompt = PromptTemplate.from_template(
    """
    Question:
    {question}
    Answer:
    {answer}
    """
)

In [24]:
prompt = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    suffix = "Quesion:\n{question}\nAnswer:",
    input_variables = [question],
    )

chain = prompt | llm

In [25]:
answer = chain.stream(
    {"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"}
)

for chunk in answer:
    print(chunk.content, end="")

이 질문에 추가 질문이 필요한가요: 예.  
추가 질문: Google은 언제 창립되었나요?  
중간 답변: Google은 1998년에 창립되었습니다.  
추가 질문: Bill Gates는 언제 태어났나요?  
중간 답변: Bill Gates는 1955년 10월 28일에 태어났습니다.  
추가 질문: 1998년에는 Bill Gates의 나이가 몇 살이었나요?  
중간 답변: 1998년에는 Bill Gates가 42세였습니다.  
최종 답변은: 42세.